In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Load data
airports_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_airports.zip"
countries_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"

airports = gpd.read_file(airports_url)
countries = gpd.read_file(countries_url)

# Task 1: Filter to African airports
# First, get African country boundaries
africa = countries[countries['CONTINENT'] == 'Africa'].copy()
print(f"African countries: {len(africa)}")

# Spatial join to find airports within Africa
african_airports = gpd.sjoin(airports, africa[['NAME', 'geometry']],
                              predicate='within')
print(f"African airports: {len(african_airports)}")

# Task 2: Reproject everything to Africa Albers (ESRI:102022) for buffering
africa_proj = africa.to_crs('ESRI:102022')
airports_proj = african_airports.to_crs('ESRI:102022')

print(f"CRS after reprojection: {airports_proj.crs}")

# Task 3: Create 100 km buffers around each airport
# YOUR CODE HERE - buffer distance is in meters (100 km = 100,000 m)
buffer_distance = 100_000  # 100 km in meters
airports_proj['buffer'] = # YOUR CODE HERE

# Create a GeoDataFrame of the buffers
buffers = gpd.GeoDataFrame(
    airports_proj[['name']],
    geometry=airports_proj['buffer'],
    crs=airports_proj.crs
)
print(f"\nCreated {len(buffers)} buffer zones")

# Task 4: Dissolve overlapping buffers into one polygon
# YOUR CODE HERE - use dissolve()
total_served_area = # YOUR CODE HERE

print(f"Dissolved into {len(total_served_area)} polygon(s)")

# Task 5: Calculate percentage of Africa covered
africa_total_area = africa_proj.geometry.area.sum()
served_area = total_served_area.geometry.area.sum()

# We need to clip the served area to Africa's boundaries
# (buffers might extend into the ocean)
africa_union = africa_proj.dissolve()
served_within_africa = total_served_area.clip(africa_union)
served_area_clipped = served_within_africa.geometry.area.sum()

percent_coverage = (served_area_clipped / africa_total_area) * 100
print(f"\nAfrica total area: {africa_total_area/1e12:.2f} million km²")
print(f"Area within 100km of airport: {served_area_clipped/1e12:.2f} million km²")
print(f"Percentage covered: {percent_coverage:.1f}%")

# Task 6: Visualize
fig, ax = plt.subplots(figsize=(12, 12))

# Plot Africa
africa_proj.plot(ax=ax, color='lightgray', edgecolor='white')

# Plot buffer zones
served_within_africa.plot(ax=ax, color='lightblue', alpha=0.5, edgecolor='blue')

# Plot airports
airports_proj.plot(ax=ax, color='red', markersize=20)

ax.set_title(f'Airport Accessibility in Africa\n{percent_coverage:.1f}% within 100km of an airport')
ax.axis('off')
plt.show()